### Allscripts Sunrise (SCM) Measurement Diagnostics

Run this notebook on a fast cluster to identify where SCM measurement rows are being filtered out.
The goal is to measure row loss across the current join and filter path without waiting on the full hydration MERGE.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur
WHERE GUID IS NOT NULL
  AND ValueText IS NOT NULL;

In [ ]:
%sql
SELECT COUNT(*) AS status_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur
WHERE GUID IS NOT NULL
  AND ValueText IS NOT NULL
  AND StatusType = 1;

In [ ]:
%sql
SELECT COUNT(*) AS numeric_rows_strict_regex
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur
WHERE GUID IS NOT NULL
  AND ValueText IS NOT NULL
  AND StatusType = 1
  AND ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$';

In [ ]:
%sql
SELECT COUNT(*) AS joined_observation_document_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
  AND obs.ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$';

In [ ]:
%sql
SELECT COUNT(*) AS joined_client_document_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
  AND obs.ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$';

In [ ]:
%sql
SELECT COUNT(*) AS joined_person_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
  AND obs.ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$';

In [ ]:
%sql
SELECT COUNT(*) AS joined_visit_rows
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON stvo.visit_occurrence_source_value = CONCAT('allscripts_scm', ' | ', CAST(doc.ClientVisitGUID AS STRING))
 AND stvo.active_flag = TRUE
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
  AND obs.ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
  AND stvo.visit_occurrence_id IS NOT NULL;

In [ ]:
%sql
SELECT
  ValueText,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur
WHERE GUID IS NOT NULL
  AND ValueText IS NOT NULL
GROUP BY ValueText
ORDER BY row_count DESC
LIMIT 50;

In [ ]:
%sql
SELECT
  ValueText,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur
WHERE GUID IS NOT NULL
  AND ValueText IS NOT NULL
  AND StatusType = 1
  AND NOT (ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$')
GROUP BY ValueText
ORDER BY row_count DESC
LIMIT 50;

In [ ]:
%sql
SELECT
  obs.GUID,
  obs.ObsItemGUID,
  obs.ValueText,
  obs.UnitOfMeasure,
  obs.StatusType,
  obs.ETL_LOAD_TS,
  obsdoc.RecordedDtm,
  doc.ClientGUID,
  doc.ClientVisitGUID,
  stp.person_id,
  stvo.visit_occurrence_id
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
LEFT JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON stvo.visit_occurrence_source_value = CONCAT('allscripts_scm', ' | ', CAST(doc.ClientVisitGUID AS STRING))
 AND stvo.active_flag = TRUE
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
ORDER BY obs.ETL_LOAD_TS DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  CAST(obs.ObsItemGUID AS STRING) AS source_id,
  COUNT(*) AS source_row_count,
  MAX(meas_concept.omop_concept_id) AS mapped_measurement_concept_id
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = CAST(obs.ObsItemGUID AS STRING)
 AND meas_concept.domain_id = 'Measurement'
 AND meas_concept.source_system = 'allscripts_scm'
WHERE obs.GUID IS NOT NULL
  AND obs.ValueText IS NOT NULL
  AND obs.StatusType = 1
GROUP BY CAST(obs.ObsItemGUID AS STRING)
ORDER BY source_row_count DESC
LIMIT 100;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW silver_measurement_diagnostic AS
SELECT
  stp.person_id,
  COALESCE(meas_concept.omop_concept_id, 0) AS measurement_concept_id,
  CAST(COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) AS DATE) AS measurement_date,
  COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) AS measurement_datetime,
  NULL AS measurement_time,
  44818702 AS measurement_type_concept_id,
  NULL AS operator_concept_id,
  CAST(obs.ValueText AS DOUBLE) AS value_as_number,
  NULL AS value_as_concept_id,
  0 AS unit_concept_id,
  NULL AS range_low,
  NULL AS range_high,
  stpr.provider_id AS provider_id,
  stvo.visit_occurrence_id AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3observationcur', 'GUID', CAST(obs.GUID AS STRING)) AS measurement_source_value,
  0 AS measurement_source_concept_id,
  obs.UnitOfMeasure AS unit_source_value,
  obs.ValueText AS value_source_value,
  'allscripts_scm' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
  ON obs.GUID = obsdoc.ObservationGUID
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
  ON obsdoc.OwnerGUID = doc.GUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_provider stpr
  ON stpr.provider_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(doc.AuthoredProviderGUID AS STRING))
 AND stpr.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON stvo.visit_occurrence_source_value = CONCAT('allscripts_scm', ' | ', CAST(doc.ClientVisitGUID AS STRING))
 AND stvo.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = CAST(obs.ObsItemGUID AS STRING)
 AND meas_concept.domain_id = 'Measurement'
 AND meas_concept.source_system = 'allscripts_scm'
WHERE obs.GUID IS NOT NULL
  AND obs.StatusType = 1
  AND obs.ValueText IS NOT NULL
  AND obs.ValueText RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
  AND COALESCE(obsdoc.RecordedDtm, doc.AuthoredDtm, obs.ETL_LOAD_TS) >= TIMESTAMP('1950-01-01');

In [ ]:
%sql
SELECT COUNT(*) AS silver_measurement_diagnostic_rows
FROM silver_measurement_diagnostic;

In [ ]:
%sql
SELECT *
FROM silver_measurement_diagnostic
LIMIT 50;